In [ ]:
############# MASCHERA GIORNO NOTTE ##############

from datetime import datetime, timedelta
import os
import numpy as np
from scipy.io import loadmat, savemat
import ephem
import re

# Cartelle
data_directory = "C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/OUTPUT_1"
mask_directory = "C:/Users/user/Documents/TESI/datigrezzi/Dati_grezzi_2020/MASCHERE"
os.makedirs(mask_directory, exist_ok=True)

# Funzioni
def calcola_fuso_orario(longitudine):
    fuso = round(longitudine / 15)
    #print(f"🕓 Fuso orario stimato per lon {longitudine}: {fuso}")
    return fuso

def get_sunrise_sunset(local_datetime, latitude, longitude):
    observer = ephem.Observer()
    observer.lat = str(latitude)
    observer.lon = str(longitude)
    observer.date = local_datetime
    sunrise = observer.previous_rising(ephem.Sun(), use_center=True)
    sunset = observer.next_setting(ephem.Sun(), use_center=True)
    alba = ephem.localtime(sunrise)
    tramonto = ephem.localtime(sunset)
    #print(f"🌄 Alba: {alba}, 🌇 Tramonto: {tramonto}")
    return alba, tramonto

def is_daytime(local_datetime, sunrise, sunset):
    daytime_start = sunrise + timedelta(hours=3)
    daytime_end = sunset - timedelta(hours=3)
    print(f"🕒 Ora locale osservazione: {local_datetime}")
    print(f"🌄 Alba: {sunrise}, 🌇 Tramonto: {sunset}")
    print(f"🕒 Periodo di giorno: {daytime_start} - {daytime_end}")
    return daytime_start <= local_datetime <= daytime_end

# Maschere
giorno_maschera = []
notte_maschera = []


def estrai_doy_time(nome_file):
    # Cerca DOYxxx_TIMEyyy nei nomi file
    match = re.search(r"DOY(\d+)_TIME(\d+)", nome_file)
    if match:
        doy = int(match.group(1))
        time = int(match.group(2))
        return doy, time
    else:
        return float('inf'), float('inf')  # Per sicurezza, sposta in fondo se non matcha

mat_files = sorted(
    [f for f in os.listdir(data_directory) if f.endswith(".mat")],
    key=estrai_doy_time
)
# File .mat

print(f"📂 Trovati {len(mat_files)} file .mat")
for file in mat_files[:10]:  # fai test su pochi file per ora
    dati = loadmat(os.path.join(data_directory, file))

    day = dati["dayOfYear"]
    time = dati["iTmOfDay"]

    print(f"\n📄 File: {file}")
    print(f"📏 dayOfYear shape: {day.shape}, iTmOfDay shape: {time.shape}")
    print("🧾 Primi 5 valori:")
    print("   dayOfYear:", day.flatten()[:5])
    print("   iTmOfDay :", time.flatten()[:5])
    
for idx, file in enumerate(mat_files):
    file_path = os.path.join(data_directory, file)
    print(f"\n📄 [{idx+1}/{len(mat_files)}] File: {file}")

    try:
        dati = loadmat(file_path)
        
        # === Debug: stampa raw dayOfYear e iTmOfDay ===
        if 'dayOfYear' in dati and 'iTmOfDay' in dati:
            day_of_year_raw = dati['dayOfYear']
            iTmOfDay_raw = dati['iTmOfDay']
        
            print(f"🧾 Raw dayOfYear: {day_of_year_raw}")
            print(f"🧾 Raw iTmOfDay: {iTmOfDay_raw}")
        
            day_of_year = int(day_of_year_raw.flatten()[0])  # Estrai il giorno dell'anno
            minute_of_day = int(iTmOfDay_raw.flatten()[0]) * 15  # 15 minuti per ogni unità
            dt_utc = datetime(2020, 1, 1) + timedelta(days=day_of_year, minutes=minute_of_day)
        
            print(f"📅 Interpreted Day: {day_of_year - 1}, ⏰ Minute of day: {minute_of_day}")
            print(f"🕒 UTC datetime: {dt_utc}")
        else:
            print(f"⚠️ dayOfYear o iTmOfDay mancanti in {file}")

        # Estrai lat/lon
        if '12' in dati and '13' in dati:
            latitudes = dati['12']  # Matricle di latitudine
            longitudes = dati['13']  # Matricle di longitudine
            print(f"🗺️ Latitudine/Longitudine estratti")
        else:
            print(f"⚠️ Latitudine/longitudine mancanti in {file}")
            giorno_maschera.extend([0] * latitudes.size)
            notte_maschera.extend([0] * latitudes.size)
            continue

        # Costruisci datetime UTC
        dt_utc = datetime(2020, 1, 1) + timedelta(days=day_of_year, minutes=minute_of_day)
        print(f"🕒 Datetime UTC: {dt_utc}")

        # Itera su tutte le latitudini e longitudini
        for lat, lon in zip(latitudes.flatten(), longitudes.flatten()):
            # Converti in locale
            fuso = calcola_fuso_orario(lon)
            dt_local = dt_utc + timedelta(hours=fuso)
            #print(f"🕒 Datetime locale: {dt_local}")

            # Calcola alba e tramonto
            alba, tramonto = get_sunrise_sunset(dt_local, lat, lon)

            # Giorno o notte?
            if is_daytime(dt_local, alba, tramonto):
                #print("🌞 È GIORNO")
                giorno_maschera.append(1)
                notte_maschera.append(0)
            else:
                #print("🌙 È NOTTE")
                giorno_maschera.append(0)
                notte_maschera.append(1)
    except Exception as e:
        print(f"❌ Errore nel file {file}: {e}")
        giorno_maschera.extend([0] * latitudes.size)
        notte_maschera.extend([0] * latitudes.size)

# Salvataggio
print("\n💾 Salvataggio delle maschere...")
savemat(os.path.join(mask_directory, 'giorno_maschera_3h.mat'), {'giorno_maschera': np.array(giorno_maschera)})
savemat(os.path.join(mask_directory, 'notte_maschera_3h.mat'), {'notte_maschera': np.array(notte_maschera)})
print("✅ Maschere salvate!")

giorno_array = np.array(giorno_maschera)
print("\n📊 Statistiche maschera giorno:")
print(f"   Totale elementi: {giorno_array.size}")
print(f"   Numero di 1 (giorno): {np.sum(giorno_array == 1)}")
print(f"   Numero di 0 (notte):  {np.sum(giorno_array == 0)}")
print(f"   Valori unici: {np.unique(giorno_array)}")

In [ ]:
############# MASCHERA GIORNO NOTTE CGE USO!!!!! ##############

from datetime import datetime, timedelta
import os
import numpy as np
from scipy.io import loadmat, savemat
import ephem
import re

# Cartelle
data_directory = "C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/OUTPUT_1"
mask_directory = "C:/Users/user/Documents/TESI/datigrezzi/Dati_grezzi_2020/MASCHERE"
os.makedirs(mask_directory, exist_ok=True)

# Funzioni
def calcola_fuso_orario(longitudine):
    return round(longitudine / 15)

def get_sunrise_sunset(dt_utc, latitude, longitude):
    observer = ephem.Observer()
    observer.lat = str(latitude)
    observer.lon = str(longitude)
    observer.date = dt_utc.date()  # Solo la data, ephem vuole un giorno intero
    observer.pressure = 0  # Disabilita correzioni atmosferiche

    # Calcola alba e tramonto (risultati in UTC)
    sunrise = observer.next_rising(ephem.Sun(), use_center=True)
    sunset = observer.next_setting(ephem.Sun(), use_center=True)
    alba = ephem.localtime(sunrise)  # Converti in datetime locale del computer
    tramonto = ephem.localtime(sunset)
    return alba, tramonto

def is_daytime(local_datetime, sunrise, sunset):
    daytime_start = sunrise + timedelta(hours=2)
    daytime_end = sunset - timedelta(hours=2)
    #print(f"🕒 Ora locale osservazione: {local_datetime}")
    #print(f"🌄 Alba: {sunrise}, 🌇 Tramonto: {sunset}")
    #print(f"🕒 Periodo di giorno: {daytime_start} - {daytime_end}")
    return daytime_start <= local_datetime <= daytime_end

def estrai_doy_time(nome_file):
    match = re.search(r"DOY(\d+)_TIME(\d+)", nome_file)
    if match:
        doy = int(match.group(1))
        time = int(match.group(2))
        return doy, time
    else:
        return float('inf'), float('inf')

# Maschere
giorno_maschera = []
notte_maschera = []

# File .mat da elaborare
mat_files = sorted(
    [f for f in os.listdir(data_directory) if f.endswith(".mat")],
    key=estrai_doy_time
)


for idx, file in enumerate(mat_files):
    file_path = os.path.join(data_directory, file)
    print(f"\n📄 [{idx+1}/{len(mat_files)}] File: {file}")

    try:
        dati = loadmat(file_path)

        if 'dayOfYear' in dati and 'iTmOfDay' in dati:
            day_of_year = int(dati['dayOfYear'].flatten()[0])
            minute_of_day = int(dati['iTmOfDay'].flatten()[0]) * 15
            dt_utc = datetime(2020, 1, 1) + timedelta(days=day_of_year, minutes=minute_of_day)
            print(f"📅 day: {day_of_year}, ⏰ minute: {minute_of_day}, 🕒 UTC datetime: {dt_utc}")
        else:
            print(f"⚠️ dayOfYear o iTmOfDay mancanti in {file}")
            continue

        if '12' in dati and '13' in dati:
            latitudes = dati['12']
            longitudes = dati['13']
            print(f"🗺️ Dimensione lat/lon: {latitudes.shape}")
        else:
            print(f"⚠️ Latitudine/longitudine mancanti in {file}")
            giorno_maschera.extend([0] * latitudes.size)
            notte_maschera.extend([0] * latitudes.size)
            continue

        for lat, lon in zip(latitudes.flatten(), longitudes.flatten()):
            fuso = calcola_fuso_orario(lon)
            dt_local = dt_utc + timedelta(hours=fuso)

            alba, tramonto = get_sunrise_sunset(dt_utc, lat, lon)

            if is_daytime(dt_local, alba, tramonto):
                giorno_maschera.append(1)
                notte_maschera.append(0)
                #print("🌞 È GIORNO")
            else:
                giorno_maschera.append(0)
                notte_maschera.append(1)
                #print("🌙 È NOTTE")

    except Exception as e:
        print(f"❌ Errore nel file {file}: {e}")
        giorno_maschera.extend([0] * latitudes.size)
        notte_maschera.extend([0] * latitudes.size)

# Salvataggio
print("\n💾 Salvataggio delle maschere...")
savemat(os.path.join(mask_directory, 'giorno_maschera_3h.mat'), {'giorno_maschera': np.array(giorno_maschera)})
savemat(os.path.join(mask_directory, 'notte_maschera_3h.mat'), {'notte_maschera': np.array(notte_maschera)})
print("✅ Maschere salvate!")

# Statistiche
giorno_array = np.array(giorno_maschera)
print("\n📊 Statistiche maschera giorno:")
print(f"   Totale elementi: {giorno_array.size}")
print(f"   Numero di 1 (giorno): {np.sum(giorno_array == 1)}")
print(f"   Numero di 0 (notte):  {np.sum(giorno_array == 0)}")
print(f"   Valori unici: {np.unique(giorno_array)}")

In [ ]:
# Carica la maschera giorno salvata
from scipy.io import loadmat

# Percorso del file dove hai salvato la maschera
mask_directory = "C:/Users/user/Documents/TESI/datigrezzi/Dati_grezzi_2020/MASCHERE"
giorno_maschera_data = loadmat(os.path.join(mask_directory, 'giorno_maschera_3h.mat'))

# Estrai la maschera giorno
giorno_maschera = giorno_maschera_data['giorno_maschera']

# Stampa i primi 50 valori della maschera giorno
print(giorno_maschera.flatten()[:50])

In [ ]:

import os
from datetime import datetime, timedelta
from scipy.io import loadmat
import numpy as np
import matplotlib.pyplot as plt
import ephem

# === CONFIG ===
data_dir = r"C:\Users\user\Documents\TESI\datigrezzi\DATI_grezzi_2020\OUTPUT_1"

def estrai_doy_time(nome_file):
    # Cerca DOYxxx_TIMEyyy nei nomi file
    match = re.search(r"DOY(\d+)_TIME(\d+)", nome_file)
    if match:
        doy = int(match.group(1))
        time = int(match.group(2))
        return doy, time
    else:
        return float('inf'), float('inf')  # Per sicurezza, sposta in fondo se non matcha

mat_files = sorted(
    [f for f in os.listdir(data_directory) if f.endswith(".mat")],
    key=estrai_doy_time
)

# === Trova due file estivi: uno alle 18 (giorno) e uno alle 22 (notte) ===
def trova_file_estate(orario_target):
    for file in mat_files:
        dati = loadmat(os.path.join(data_dir, file))
        if "dayOfYear" in dati and "iTmOfDay" in dati:
            giorno = int(dati["dayOfYear"].flatten()[0])
            ora = int(dati["iTmOfDay"].flatten()[0]) * 15 // 60
            if 170 <= giorno <= 230 and ora == orario_target:
                return file
    return None

file_giorno = trova_file_estate(15)
file_notte = trova_file_estate(23)

print(f"🌞 File giorno: {file_giorno}")
print(f"🌙 File notte: {file_notte}")

# === Funzioni per calcolo giorno/notte ===
def calcola_fuso_orario(lon):
    return round(lon / 15)

def get_sunrise_sunset(local_dt, lat, lon):
    observer = ephem.Observer()
    observer.lat = str(lat)
    observer.lon = str(lon)
    observer.date = local_dt
    alba = ephem.localtime(observer.previous_rising(ephem.Sun(), use_center=True))
    tramonto = ephem.localtime(observer.next_setting(ephem.Sun(), use_center=True))
    return alba, tramonto

def is_daytime(local_dt, alba, tramonto):
    return alba <= local_dt <= tramonto

# === Visualizza classificazione giorno/notte ===
def mostra_classificazione(file, titolo):
    dati = loadmat(os.path.join(data_dir, file))
    lat = dati["12"].flatten()
    lon = dati["13"].flatten()
    day = int(dati["dayOfYear"].flatten()[0])
    min_day = int(dati["iTmOfDay"].flatten()[0]) * 15
    dt_utc = datetime(2020, 1, 1) + timedelta(days=day, minutes=min_day)

    giorno_mask = []
    notte_mask = []

    for la, lo in zip(lat, lon):
        fuso = calcola_fuso_orario(lo)
        dt_local = dt_utc + timedelta(hours=fuso)
        alba, tramonto = get_sunrise_sunset(dt_local, la, lo)
        if is_daytime(dt_local, alba, tramonto):
            giorno_mask.append(True)
            notte_mask.append(False)
        else:
            giorno_mask.append(False)
            notte_mask.append(True)

    plt.figure(figsize=(7, 6))
    plt.scatter(lon[giorno_mask], lat[giorno_mask], color="gold", label="Giorno", s=10)
    plt.scatter(lon[notte_mask], lat[notte_mask], color="darkblue", label="Notte", s=10)
    plt.xlabel("Longitudine")
    plt.ylabel("Latitudine")
    plt.title(titolo)
    plt.legend()
    plt.grid(True)
    plt.show()

# === Grafico alba/tramonto vs latitudine in estate ===
def curva_alba_tramonto():
    latitudini = np.linspace(35, 55, 100)
    giorno = 196
    alba = []
    tramonto = []

    for lat in latitudini:
        obs = ephem.Observer()
        obs.lat = str(lat)
        obs.lon = "15"  # longitudine media Italia
        obs.date = datetime(2020, 1, 1) + timedelta(days=giorno - 1)
        alba.append(ephem.localtime(obs.previous_rising(ephem.Sun())).hour)
        tramonto.append(ephem.localtime(obs.next_setting(ephem.Sun())).hour)

    plt.figure(figsize=(7, 4))
    plt.plot(latitudini, alba, label="Alba", color="orange")
    plt.plot(latitudini, tramonto, label="Tramonto", color="red")
    plt.fill_between(latitudini, alba, tramonto, color="gold", alpha=0.3)
    plt.xlabel("Latitudine")
    plt.ylabel("Ora locale")
    plt.title("Alba e Tramonto al Giorno 196 (15 Luglio)")
    plt.legend()
    plt.grid(True)
    plt.show()

# === Esecuzione ===
if file_giorno:
    mostra_classificazione(file_giorno, "Distribuzione Giorno - Estate ore 15")
if file_notte:
    mostra_classificazione(file_notte, "Distribuzione Notte - Estate ore 23")

curva_alba_tramonto()


In [4]:
##################### MASCHERA STAGIONI #####################

import os
import numpy as np
from scipy.io import loadmat, savemat
from datetime import datetime
import random
import re

# Directory di input/output
data_directory = "C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/OUTPUT_1"
mask_directory = "C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/MASCHERE"
os.makedirs(mask_directory, exist_ok=True)

# Funzione per determinare la stagione in base al dayOfYear
def get_season_by_day_of_year(day_of_year):
    if 335 <= day_of_year <= 365 or 1 <= day_of_year <= 59:
        return 1  # Inverno
    elif 152 <= day_of_year <= 243:
        return 0  # Estate
    else:
        return 2  # Primavera o Autunno (da escludere)

# Inizializza la maschera per estate, inverno e altre stagioni
stagionalita_maschera = []

# Trova tutti i file .mat nella cartella OUTPUT_1
def estrai_doy_time(nome_file):
    # Cerca DOYxxx_TIMEyyy nei nomi file
    match = re.search(r"DOY(\d+)_TIME(\d+)", nome_file)
    if match:
        doy = int(match.group(1))
        time = int(match.group(2))
        return doy, time
    else:
        return float('inf'), float('inf')  # Per sicurezza, sposta in fondo se non matcha

mat_files = sorted(
    [f for f in os.listdir(data_directory) if f.endswith(".mat")],
    key=estrai_doy_time
)

if not mat_files:
    print("⚠️ Nessun file .mat trovato nella cartella OUTPUT_1!")
else:
    print(f"📂 Trovati {len(mat_files)} file da elaborare in {data_directory}")

# Elaborazione dei file .mat
for file_name in mat_files:
    file_path = os.path.join(data_directory, file_name)
    print(f"\n📄 Elaboro file: {file_name}")
    
    try:
        dati = loadmat(file_path)

        if '12' in dati and '13' in dati and dati['12'].size > 0 and dati['13'].size > 0:
            latitudine = dati['12'].flatten()
            longitudine = dati['13'].flatten()
            day_of_year = int(dati['dayOfYear'][0])
            print(f"🔍 Prima latitudine: {latitudine[0]}, Prima longitudine: {longitudine[0]}")

            sample_indices = random.sample(range(len(latitudine)), min(5, len(latitudine)))

            print(f"🗺️ Elaboro {len(latitudine)} punti geografici...")

            for i, (lat, lon) in enumerate(zip(latitudine, longitudine)):
                try:
                    # Determina la stagione usando il dayOfYear
                    stagione = get_season_by_day_of_year(day_of_year)

                    # Aggiungi il valore della stagione alla maschera
                    stagionalita_maschera.append(stagione)

                    # Stampa solo per alcuni punti casuali
                    if i in sample_indices:
                        print(f"🌍 Lat: {lat}, Lon: {lon}, Stagione: {stagione}")

                    if len(stagionalita_maschera) % 1000 == 0:
                        print(f"✅ Elaborati {len(stagionalita_maschera)} punti...")

                except Exception as e:
                    print(f"⚠️ Errore con lat={lat}, lon={lon}: {e}")
                    stagionalita_maschera.append(2)  # 2 per errore (altre stagioni)

        else:
            print(f"⚠️ File {file_name} non contiene dati validi nei canali di latitudine (12) o longitudine (13)")

    except Exception as e:
        print(f"❌ Errore nel caricamento di {file_name}: {e}")

# Salvataggio della maschera finale
print("\n💾 Salvataggio della maschera stagionale in formato .mat...")

stagionalita_dict = {"stagionalita_maschera": np.array(stagionalita_maschera)}
output_path = os.path.join(mask_directory, 'stagionalita_maschera_estate_inverno.mat')
savemat(output_path, stagionalita_dict)

print(f"✅ Maschera stagionale salvata con successo in '{output_path}'!")

📂 Trovati 1440 file da elaborare in C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/OUTPUT_1

📄 Elaboro file: msgDprImerg_CalFlipCln_2020_DOY0_TIME12.mat
🔍 Prima latitudine: 45.20669937133789, Prima longitudine: 6.977297306060791
🗺️ Elaboro 4096 punti geografici...
🌍 Lat: 47.323768615722656, Lon: 7.582963466644287, Stagione: 2
🌍 Lat: 46.20947265625, Lon: 7.658308982849121, Stagione: 2
✅ Elaborati 1000 punti...
🌍 Lat: 45.74832534790039, Lon: 8.28699016571045, Stagione: 2
✅ Elaborati 2000 punti...
🌍 Lat: 47.60794448852539, Lon: 9.171213150024414, Stagione: 2
✅ Elaborati 3000 punti...
🌍 Lat: 47.66916275024414, Lon: 9.613818168640137, Stagione: 2
✅ Elaborati 4000 punti...

📄 Elaboro file: msgDprImerg_CalFlipCln_2020_DOY0_TIME18.mat
🔍 Prima latitudine: 34.83046340942383, Prima longitudine: -6.894141674041748
🗺️ Elaboro 4096 punti geografici...
✅ Elaborati 5000 punti...
🌍 Lat: 36.014732360839844, Lon: -6.315855979919434, Stagione: 2
🌍 Lat: 35.525146484375, Lon: -6.168262958526611, S

🔍 Prima latitudine: 35.84583282470703, Prima longitudine: 4.779650688171387
🗺️ Elaboro 4096 punti geografici...
🌍 Lat: 35.960533142089844, Lon: 4.960372447967529, Stagione: 1
🌍 Lat: 36.91112518310547, Lon: 5.029950141906738, Stagione: 1
✅ Elaborati 91000 punti...
🌍 Lat: 36.002071380615234, Lon: 5.309417724609375, Stagione: 1
🌍 Lat: 36.99372100830078, Lon: 5.527891635894775, Stagione: 1
✅ Elaborati 92000 punti...
🌍 Lat: 36.202789306640625, Lon: 6.263497352600098, Stagione: 1
✅ Elaborati 93000 punti...
✅ Elaborati 94000 punti...

📄 Elaboro file: msgDprImerg_CalFlipCln_2020_DOY5_TIME74.mat
🔍 Prima latitudine: 43.816749572753906, Prima longitudine: -11.737639427185059
🗺️ Elaboro 4096 punti geografici...
✅ Elaborati 95000 punti...
🌍 Lat: 44.79981231689453, Lon: -11.224687576293945, Stagione: 1
✅ Elaborati 96000 punti...
🌍 Lat: 43.961692810058594, Lon: -10.520861625671387, Stagione: 1
🌍 Lat: 46.67758560180664, Lon: -10.979935646057129, Stagione: 1
✅ Elaborati 97000 punti...
🌍 Lat: 45.4635887

🔍 Prima latitudine: 44.46398162841797, Prima longitudine: 8.88446044921875
🗺️ Elaboro 4096 punti geografici...
🌍 Lat: 46.43936538696289, Lon: 9.325909614562988, Stagione: 1
✅ Elaborati 185000 punti...
🌍 Lat: 46.987003326416016, Lon: 9.900300025939941, Stagione: 1
🌍 Lat: 45.790977478027344, Lon: 10.032529830932617, Stagione: 1
✅ Elaborati 186000 punti...
🌍 Lat: 46.045387268066406, Lon: 10.669873237609863, Stagione: 1
✅ Elaborati 187000 punti...
🌍 Lat: 46.21438217163086, Lon: 11.54881477355957, Stagione: 1
✅ Elaborati 188000 punti...

📄 Elaboro file: msgDprImerg_CalFlipCln_2020_DOY11_TIME5.mat
🔍 Prima latitudine: 40.50188446044922, Prima longitudine: -10.073709487915039
🗺️ Elaboro 4096 punti geografici...
🌍 Lat: 41.33576583862305, Lon: -9.989930152893066, Stagione: 1
✅ Elaborati 189000 punti...
🌍 Lat: 42.40591812133789, Lon: -10.064935684204102, Stagione: 1
✅ Elaborati 190000 punti...
🌍 Lat: 43.036521911621094, Lon: -9.079873085021973, Stagione: 1
✅ Elaborati 191000 punti...
🌍 Lat: 40.63

🔍 Prima latitudine: 38.62321853637695, Prima longitudine: -7.438604831695557
🗺️ Elaboro 4096 punti geografici...
✅ Elaborati 279000 punti...
🌍 Lat: 40.73578643798828, Lon: -7.363968849182129, Stagione: 1
🌍 Lat: 41.02743148803711, Lon: -7.325466632843018, Stagione: 1
🌍 Lat: 39.25534439086914, Lon: -6.965818881988525, Stagione: 1
✅ Elaborati 280000 punti...
✅ Elaborati 281000 punti...
🌍 Lat: 39.76296615600586, Lon: -5.699862003326416, Stagione: 1
🌍 Lat: 41.129390716552734, Lon: -5.832380294799805, Stagione: 1
✅ Elaborati 282000 punti...

📄 Elaboro file: msgDprImerg_CalFlipCln_2020_DOY30_TIME37.mat
🔍 Prima latitudine: 34.11012649536133, Prima longitudine: 9.71939468383789
🗺️ Elaboro 4096 punti geografici...
✅ Elaborati 283000 punti...
🌍 Lat: 36.35381317138672, Lon: 10.422842979431152, Stagione: 1
🌍 Lat: 35.83056640625, Lon: 10.659951210021973, Stagione: 1
✅ Elaborati 284000 punti...
🌍 Lat: 36.10515213012695, Lon: 11.127304077148438, Stagione: 1
✅ Elaborati 285000 punti...
🌍 Lat: 35.887546

🔍 Prima latitudine: 45.058204650878906, Prima longitudine: 23.362712860107422
🗺️ Elaboro 4096 punti geografici...
✅ Elaborati 377000 punti...
🌍 Lat: 45.437095642089844, Lon: 24.280750274658203, Stagione: 1
✅ Elaborati 378000 punti...
🌍 Lat: 46.801490783691406, Lon: 25.633777618408203, Stagione: 1
✅ Elaborati 379000 punti...
🌍 Lat: 46.39497375488281, Lon: 26.019681930541992, Stagione: 1
🌍 Lat: 45.23756790161133, Lon: 25.557907104492188, Stagione: 1
✅ Elaborati 380000 punti...
🌍 Lat: 45.888824462890625, Lon: 26.25012969970703, Stagione: 1

📄 Elaboro file: msgDprImerg_CalFlipCln_2020_DOY36_TIME35.mat
🔍 Prima latitudine: 36.434444427490234, Prima longitudine: -6.564263820648193
🗺️ Elaboro 4096 punti geografici...
✅ Elaborati 381000 punti...
🌍 Lat: 38.25360107421875, Lon: -6.604067802429199, Stagione: 1
🌍 Lat: 38.52970886230469, Lon: -6.5606160163879395, Stagione: 1
🌍 Lat: 37.66086196899414, Lon: -6.259108066558838, Stagione: 1
✅ Elaborati 382000 punti...
🌍 Lat: 38.43962478637695, Lon: -5.7

...

🔍 Prima latitudine: 48.32386016845703, Prima longitudine: 7.574460029602051
🗺️ Elaboro 4096 punti geografici...
✅ Elaborati 5522000 punti...
🌍 Lat: 49.73948287963867, Lon: 8.26695442199707, Stagione: 1
✅ Elaborati 5523000 punti...
✅ Elaborati 5524000 punti...
🌍 Lat: 49.61602783203125, Lon: 9.633922576904297, Stagione: 1
✅ Elaborati 5525000 punti...
🌍 Lat: 49.0560417175293, Lon: 10.17879867553711, Stagione: 1
🌍 Lat: 51.61777877807617, Lon: 11.0014066696167, Stagione: 1
🌍 Lat: 48.75075912475586, Lon: 10.330419540405273, Stagione: 1

📄 Elaboro file: msgDprImerg_CalFlipCln_2020_DOY343_TIME89.mat
🔍 Prima latitudine: 34.840660095214844, Prima longitudine: 20.42237091064453
🗺️ Elaboro 4096 punti geografici...
🌍 Lat: 37.105472564697266, Lon: 21.2918701171875, Stagione: 1
✅ Elaborati 5526000 punti...
✅ Elaborati 5527000 punti...
🌍 Lat: 36.11442184448242, Lon: 22.191272735595703, Stagione: 1
✅ Elaborati 5528000 punti...
🌍 Lat: 36.024391174316406, Lon: 22.700586318969727, Stagione: 1
✅ Elaborati 

🔍 Prima latitudine: 48.47588348388672, Prima longitudine: -9.652819633483887
🗺️ Elaboro 4096 punti geografici...
✅ Elaborati 5628000 punti...
✅ Elaborati 5629000 punti...
🌍 Lat: 50.83480453491211, Lon: -8.94243049621582, Stagione: 1
🌍 Lat: 49.58666229248047, Lon: -8.506425857543945, Stagione: 1
✅ Elaborati 5630000 punti...
🌍 Lat: 49.8890495300293, Lon: -7.8455810546875, Stagione: 1
✅ Elaborati 5631000 punti...
🌍 Lat: 49.98137664794922, Lon: -7.187252044677734, Stagione: 1
🌍 Lat: 51.351654052734375, Lon: -7.423974514007568, Stagione: 1
✅ Elaborati 5632000 punti...

📄 Elaboro file: msgDprImerg_CalFlipCln_2020_DOY350_TIME50.mat
🔍 Prima latitudine: 34.91731643676758, Prima longitudine: -4.989896297454834
🗺️ Elaboro 4096 punti geografici...
🌍 Lat: 36.715476989746094, Lon: -4.630976676940918, Stagione: 1
✅ Elaborati 5633000 punti...
🌍 Lat: 36.068363189697266, Lon: -4.448981285095215, Stagione: 1
🌍 Lat: 36.750614166259766, Lon: -4.31895112991333, Stagione: 1
✅ Elaborati 5634000 punti...
🌍 Lat

🔍 Prima latitudine: 35.03882598876953, Prima longitudine: -0.2036360502243042
🗺️ Elaboro 4096 punti geografici...
🌍 Lat: 37.30646896362305, Lon: -0.10521575063467026, Stagione: 1
✅ Elaborati 5735000 punti...
✅ Elaborati 5736000 punti...
✅ Elaborati 5737000 punti...
✅ Elaborati 5738000 punti...
🌍 Lat: 36.08780288696289, Lon: 1.7227493524551392, Stagione: 1
🌍 Lat: 36.12554931640625, Lon: 1.7236889600753784, Stagione: 1
🌍 Lat: 37.0784797668457, Lon: 1.748002052307129, Stagione: 1
🌍 Lat: 36.9249267578125, Lon: 1.7789026498794556, Stagione: 1

📄 Elaboro file: msgDprImerg_CalFlipCln_2020_DOY356_TIME49.mat
🔍 Prima latitudine: 48.16608810424805, Prima longitudine: -11.204048156738281
🗺️ Elaboro 4096 punti geografici...
🌍 Lat: 48.67864990234375, Lon: -11.331933975219727, Stagione: 1
✅ Elaborati 5739000 punti...
🌍 Lat: 48.92243576049805, Lon: -10.90379810333252, Stagione: 1
🌍 Lat: 48.3919792175293, Lon: -10.292365074157715, Stagione: 1
🌍 Lat: 49.37746047973633, Lon: -10.477557182312012, Stagione

🔍 Prima latitudine: 34.03451156616211, Prima longitudine: -4.55991792678833
🗺️ Elaboro 4096 punti geografici...
🌍 Lat: 34.25168228149414, Lon: -4.438398361206055, Stagione: 1
✅ Elaborati 5833000 punti...
🌍 Lat: 34.06930160522461, Lon: -4.393836975097656, Stagione: 1
✅ Elaborati 5834000 punti...
🌍 Lat: 35.38913345336914, Lon: -3.9266843795776367, Stagione: 1
✅ Elaborati 5835000 punti...
🌍 Lat: 35.75833511352539, Lon: -3.3627426624298096, Stagione: 1
✅ Elaborati 5836000 punti...
🌍 Lat: 34.86099624633789, Lon: -2.5403950214385986, Stagione: 1

📄 Elaboro file: msgDprImerg_CalFlipCln_2020_DOY361_TIME70.mat
🔍 Prima latitudine: 49.38080596923828, Prima longitudine: -1.579970121383667
🗺️ Elaboro 4096 punti geografici...
✅ Elaborati 5837000 punti...
🌍 Lat: 52.583946228027344, Lon: -1.4207152128219604, Stagione: 1
🌍 Lat: 51.048160552978516, Lon: -0.8209697008132935, Stagione: 1
✅ Elaborati 5838000 punti...
🌍 Lat: 50.77167892456055, Lon: -0.1812387853860855, Stagione: 1
🌍 Lat: 52.34852600097656, 

✅ Maschera stagionale salvata con successo in 'C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/MASCHERE\stagionalita_maschera_estate_inverno.mat'!


In [ ]:
print(f"\n🔍 Dimensione della maschera stagionale: {len(stagionalita_maschera)}")

In [5]:
 ####### MASCHERA TERRA - MARE #######
    
import h5py
import numpy as np
import scipy.io as sio
from scipy.interpolate import griddata

def carica_sealand_mask(netcdf_file):
    """Carica la maschera Mare/Terra dal file NetCDF."""
    print("Caricamento della maschera Mare/Terra dal file NetCDF...")
    with h5py.File(netcdf_file, 'r') as f:
        # Estrai latitudine, longitudine e maschera
        latitudes = np.array(f['latitude'][:])  # Latitudine
        longitudes = np.array(f['longitude'][:])  # Longitudine
        sealand_mask = np.array(f['lsm'][0, :, :])  # Maschera Mare/Terra (lsm)
        
    print("Maschera Mare/Terra caricata con successo.")
    return latitudes, longitudes, sealand_mask

def regridding(latitudes_dati, longitudes_dati, mask, latitudes_mask, longitudes_mask):
    """Esegui il regridding della maschera Mare/Terra sui dati."""
    print("Avvio del processo di regridding...")
    
    # Creiamo una griglia 2D delle coordinate latitudine/longitudine della maschera
    grid_x, grid_y = np.meshgrid(longitudes_mask, latitudes_mask)
    
    # Appiattiamo le griglie per applicare griddata
    points = np.array([grid_x.flatten(), grid_y.flatten()]).T
    mask_values = mask.flatten()

    # Eseguiamo l'interpolazione bilineare
    grid_z = griddata(points, mask_values, (longitudes_dati, latitudes_dati), method='nearest')
    
    print("Regridding completato.")
    return grid_z

def salva_mascherati(latitudes, longitudes, mare_mask, terra_mask, output_dir):
    """Salva le maschere Mare e Terra come file .mat."""
    print("Salvataggio delle maschere Mare e Terra...")
    sio.savemat(f'{output_dir}/Mare_mask_regridded.mat', {'mask': mare_mask, 'latitudes': latitudes, 'longitudes': longitudes})
    sio.savemat(f'{output_dir}/Terra_mask_regridded.mat', {'mask': terra_mask, 'latitudes': latitudes, 'longitudes': longitudes})
    print("Maschere salvate con successo.")

# Carica i dati dal file .mat
# Nuova parte: caricamento di tutti i file .mat nella cartella
print("Caricamento dei dati da tutti i file .mat nella cartella...")
cartella_dati = 'C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/OUTPUT_1'

# Inizializza array vuoti per latitudine e longitudine
latitudes_list = []
longitudes_list = []

def estrai_doy_time(nome_file):
    match = re.search(r"DOY(\d+)_TIME(\d+)", nome_file)
    if match:
        doy = int(match.group(1))
        time = int(match.group(2))
        return doy, time
    return float('inf'), float('inf')  # Per evitare problemi nei file senza match

# Ordina i file per DOY e TIME estratti dal nome
file_ordinati = sorted(
    [f for f in os.listdir(cartella_dati) if f.endswith('.mat')],
    key=estrai_doy_time
)

# Liste per salvare latitudine e longitudine
latitudes_list = []
longitudes_list = []

# Scorri i file in ordine e carica lat/lon
for filename in file_ordinati:
    filepath = os.path.join(cartella_dati, filename)
    print(f"Caricando file: {filename}")
    dati_mat = sio.loadmat(filepath)
    
    # Estrai e appiattisci latitudine e longitudine
    lat = np.array(dati_mat['12']).flatten()
    lon = np.array(dati_mat['13']).flatten()
    
    latitudes_list.append(lat)
    longitudes_list.append(lon)

# Concatena tutti i dati in un unico array
latitudes_dati = np.concatenate(latitudes_list)
longitudes_dati = np.concatenate(longitudes_list)

print("Tutti i dati caricati e concatenati con successo.")

# Specifica il file NetCDF per la maschera
netcdf_file = 'C:/Users/user/Documents/TESI/datigrezzi/sea_land.nc'
output_dir = 'C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/MASCHERE'

# Carica la maschera Mare/Terra
print("Caricamento della maschera Mare/Terra dal NetCDF...")
latitudes_mask, longitudes_mask, sealand_mask = carica_sealand_mask(netcdf_file)

# Eseguiamo il regridding della maschera
print("Esecuzione del regridding della maschera Mare/Terra...")
regridded_mask = regridding(latitudes_dati, longitudes_dati, sealand_mask, latitudes_mask, longitudes_mask)

# Crea le maschere Mare e Terra dalla maschera regradata
print("Creazione delle maschere Mare e Terra...")
mare_mask = (regridded_mask == 0).astype(int)  # Mare è rappresentato da 0
terra_mask = (regridded_mask == 1).astype(int)  # Terra è rappresentata da 1

# Salviamo le maschere Mare e Terra
print("Salvataggio delle maschere Mare e Terra...")
salva_mascherati(latitudes_dati, longitudes_dati, mare_mask, terra_mask, output_dir)

print("Processo completato con successo!")

Caricamento dei dati da tutti i file .mat nella cartella...
Caricando file: msgDprImerg_CalFlipCln_2020_DOY0_TIME12.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY0_TIME18.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY0_TIME78.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY1_TIME8.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY1_TIME68.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY1_TIME69.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY1_TIME75.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY2_TIME11.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY2_TIME65.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY2_TIME71.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY2_TIME72.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY3_TIME8.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY3_TIME14.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY3_TIME68.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY3_TIME74.mat
Caricando file: msgDprImerg_CalFlipCln_202

Caricando file: msgDprImerg_CalFlipCln_2020_DOY85_TIME64.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY85_TIME70.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY85_TIME71.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY86_TIME7.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY86_TIME13.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY86_TIME67.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY87_TIME3.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY87_TIME10.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY87_TIME63.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY87_TIME64.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY87_TIME70.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY88_TIME6.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY88_TIME12.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY88_TIME66.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY88_TIME67.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY89_TIME3.mat
Caricando file: msgDprImerg_

Caricando file: msgDprImerg_CalFlipCln_2020_DOY120_TIME25.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY120_TIME31.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY120_TIME37.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY120_TIME63.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY121_TIME28.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY121_TIME34.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY121_TIME60.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY121_TIME66.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY122_TIME56.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY122_TIME62.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY122_TIME63.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY123_TIME21.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY123_TIME27.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY123_TIME59.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY123_TIME65.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY124_TIME24.mat
Caricand

Caricando file: msgDprImerg_CalFlipCln_2020_DOY188_TIME83.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY189_TIME41.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY189_TIME47.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY189_TIME79.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY190_TIME44.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY190_TIME76.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY190_TIME82.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY191_TIME40.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY191_TIME47.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY191_TIME72.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY191_TIME79.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY192_TIME37.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY192_TIME43.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY192_TIME75.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY193_TIME40.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY193_TIME72.mat
Caricand

Caricando file: msgDprImerg_CalFlipCln_2020_DOY268_TIME81.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY268_TIME87.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY269_TIME45.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY269_TIME51.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY269_TIME77.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY269_TIME83.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY269_TIME84.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY270_TIME41.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY270_TIME48.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY270_TIME54.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY270_TIME80.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY271_TIME44.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY271_TIME51.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY271_TIME76.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY271_TIME77.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY271_TIME83.mat
Caricand

Caricando file: msgDprImerg_CalFlipCln_2020_DOY298_TIME13.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY298_TIME19.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY298_TIME45.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY298_TIME51.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY298_TIME52.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY299_TIME9.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY299_TIME10.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY299_TIME16.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY299_TIME42.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY299_TIME48.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY300_TIME6.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY300_TIME12.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY300_TIME19.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY300_TIME45.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY300_TIME51.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY301_TIME9.mat
Caricando f

Caricando file: msgDprImerg_CalFlipCln_2020_DOY334_TIME62.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY334_TIME69.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY335_TIME5.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY335_TIME11.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY336_TIME1.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY336_TIME2.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY336_TIME8.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY336_TIME62.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY336_TIME68.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY336_TIME94.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY337_TIME4.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY337_TIME58.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY337_TIME65.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY337_TIME71.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY338_TIME1.mat
Caricando file: msgDprImerg_CalFlipCln_2020_DOY338_TIME7.mat
Caricando file:

Tutti i dati caricati e concatenati con successo.
Caricamento della maschera Mare/Terra dal NetCDF...
Caricamento della maschera Mare/Terra dal file NetCDF...
Maschera Mare/Terra caricata con successo.
Esecuzione del regridding della maschera Mare/Terra...
Avvio del processo di regridding...
Regridding completato.
Creazione delle maschere Mare e Terra...
Salvataggio delle maschere Mare e Terra...
Salvataggio delle maschere Mare e Terra...
Maschere salvate con successo.
Processo completato con successo!


In [ ]:
import os
import scipy.io as sio

# Parametri
cartella = r'C:\Users\user\Documents\TESI\datigrezzi\DATI_grezzi_2020\OUTPUT_1'

# Funzione per verificare le dimensioni delle chiavi
def controlla_dimensioni(file_path):
    data = sio.loadmat(file_path)
    
    # Otteniamo tutte le chiavi tranne dayOfYear e iTmOfDay
    chiavi_da_controllare = [key for key in data.keys() if key not in ['dayOfYear', 'iTmOfDay', '__header__', '__version__', '__globals__']]
    
    # Prendiamo la forma della prima chiave come riferimento
    dimensione_riferimento = data[chiavi_da_controllare[0]].shape

    # Verifica che tutte le altre chiavi abbiano la stessa dimensione
    for key in chiavi_da_controllare:
        if data[key].shape != dimensione_riferimento:
            print(f"⚠️ La chiave {key} ha dimensione {data[key].shape} (aspettato: {dimensione_riferimento})")
        else:
            print(f"La chiave {key} ha la dimensione corretta: {data[key].shape}")
    
# Controlla tutte le dimensioni dei file .mat nella cartella
for nome_file in os.listdir(cartella):
    if nome_file.endswith('.mat'):
        path = os.path.join(cartella, nome_file)
        print(f"\nControllando file: {nome_file}")
        controlla_dimensioni(path)

In [ ]:
import os
import scipy.io
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

# Funzione per caricare, riscalare l'immagine e salvarla
def processa_file(file_path, output_dir):
    # Carica il file .mat
    mat_data = scipy.io.loadmat(file_path)
    
    # Prende i canali dal file (es. chiave 1-13 per i canali e dpr)
    for i in range(1, 14):
        chiave = f'{i}'
        
        if chiave in mat_data:
            data = mat_data[chiave]
            print(f"La chiave {chiave} ha la dimensione {data.shape}")
            
            # Se la dimensione è 4096, la convertiamo in 64x64
            if data.shape[1] == 4096:
                data_resized = np.reshape(data, (64, 64))
                mat_data[chiave] = data_resized
                print(f"La chiave {chiave} è stata ridimensionata a 64x64")
                
                # Visualizza l'immagine
                plt.imshow(data_resized, cmap='viridis', norm=Normalize(vmin=np.min(data_resized), vmax=np.max(data_resized)))
                plt.colorbar()
                plt.title(f"Immagine {chiave}")
                
                # Salva l'immagine
                img_output_path = os.path.join(output_dir, f"{os.path.basename(file_path).replace('.mat', '')}_{chiave}.png")
                plt.savefig(img_output_path)
                plt.close()  # Chiude la figura dopo il salvataggio
                print(f"Immagine salvata come {img_output_path}")
            
            # Salva il file modificato
            scipy.io.savemat(file_path, mat_data)
        else:
            print(f"La chiave {chiave} non è presente nel file")

# Percorso della cartella contenente i file .mat
folder_path = r'C:\Users\user\Documents\TESI\datigrezzi\DATI_grezzi_2020\OUTPUT_1'
output_dir = r'C:\Users\user\Documents\TESI\datigrezzi\DATI_grezzi_2020\OUTPUT_1\Immagini'  # Dove salvare le immagini

# Crea la cartella per le immagini se non esiste
os.makedirs(output_dir, exist_ok=True)

# Lista dei file .mat nella cartella
file_list = [f for f in os.listdir(folder_path) if f.endswith('.mat')]

# Processa ogni file
for file_name in file_list:
    file_path = os.path.join(folder_path, file_name)
    print(f"Elaborando il file: {file_name}")
    processa_file(file_path, output_dir)

print("Elaborazione e salvataggio delle immagini completati!")

In [ ]:
import os
import scipy.io as sio
import matplotlib.pyplot as plt
import numpy as np

# Cartella contenente i file
cartella = r'C:\Users\user\Documents\TESI\datigrezzi\DATI_grezzi_2020\OUTPUT_1'

import re

def estrai_doy_time(nome_file):
    match = re.search(r"DOY(\d+)_TIME(\d+)", nome_file)
    if match:
        doy = int(match.group(1))
        time = int(match.group(2))
        return doy, time
    return float('inf'), float('inf')  # metti alla fine se non trova numeri

file_trovato = None

# Ordina i file per giorno e ora
file_ordinati = sorted(
    [f for f in os.listdir(cartella) if f.endswith('.mat')],
    key=estrai_doy_time
)

for nome_file in file_ordinati:
    path = os.path.join(cartella, nome_file)
    try:
        data = sio.loadmat(path)

        canali_completi = True
        for i in range(1, 12):
            if data[str(i)].flatten().shape[0] != 4096:
                canali_completi = False
                break

        if canali_completi:
            file_trovato = path
            print(f"✅ File trovato: {nome_file}")
            break
    except Exception as e:
        print(f"Errore nel file {nome_file}: {e}")

if file_trovato is None:
    print("❌ Nessun file trovato con 4096 elementi per ogni canale.")
else:
    print(f"✅ File selezionato: {file_trovato}")
    data = sio.loadmat(file_trovato)

    # Estrae la data e l'ora per il primo indice del file
    giorno = int(data['dayOfYear'].flatten()[0])
    ora = int(data['iTmOfDay'].flatten()[0])
    print(f"Giorno: {giorno}, Ora (slot): {ora} → {(ora * 15) // 60}:{(ora * 15) % 60:02d}")

    # Lat e Lon alla posizione selezionata (valori 1D)
    lat = data['12'].flatten()
    lon = data['13'].flatten()

    # Gestione dei NaN (se ci sono)
    lat = np.nan_to_num(lat, nan=np.nanmean(lat))  # Sostituisce i NaN con la media di latitudine
    lon = np.nan_to_num(lon, nan=np.nanmean(lon))  # Sostituisce i NaN con la media di longitudine

    # Reshape latitudine e longitudine in 64x64 (come le immagini)
    lat_grid = lat.reshape(64, 64)
    lon_grid = lon.reshape(64, 64)

    # Rigrida i dati per ogni canale
    fig, axs = plt.subplots(3, 4, figsize=(16, 12))
    axs = axs.flatten()

    for i in range(1, 12):  # canali 1-11
        canale = data[str(i)].flatten()  # Assicurati che il canale sia 1D
        canale_reshape = canale.reshape(64, 64)  # Reshape a 64x64 (rigridamento)

        # Plot del canale
        im = axs[i-1].imshow(canale_reshape, cmap='bone',
                             extent=[np.min(lon_grid), np.max(lon_grid), np.min(lat_grid), np.max(lat_grid)])
        axs[i-1].set_title(f'Canale {i}')
        axs[i-1].set_xlabel('Longitudine')
        axs[i-1].set_ylabel('Latitudine')
        fig.colorbar(im, ax=axs[i-1], orientation='vertical', shrink=0.7)

    axs[-1].axis('off')  # Rimuovi subplot vuoto
    plt.tight_layout()
    plt.show()

In [ ]:
import os
import scipy.io as sio
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patches as patches  # Per disegnare il rettangolo

# === PARAMETRI ===
cartella = r'C:\Users\user\Documents\TESI\datigrezzi\DATI_grezzi_2020\OUTPUT_1'
maschera_path = r'C:\Users\user\Documents\TESI\datigrezzi\DATI_grezzi_2020\MASCHERE\giorno_maschera_3h.mat'

# === CERCA IL PRIMO FILE CON 4096 ELEMENTI PER CANALE ===
import re

def estrai_doy_time(nome_file):
    match = re.search(r"DOY(\d+)_TIME(\d+)", nome_file)
    if match:
        doy = int(match.group(1))
        time = int(match.group(2))
        return doy, time
    return float('inf'), float('inf')  # Restituisce valori molto alti se non trova match

file_trovato = None

# Ordina i file per DOY e TIME
file_ordinati = sorted(
    [f for f in os.listdir(cartella) if f.endswith('.mat')],
    key=estrai_doy_time
)

# Scorri i file ordinati
for nome_file in file_ordinati:
    path = os.path.join(cartella, nome_file)
    try:
        data = sio.loadmat(path)

        # Verifica se tutti i canali sono completi (lunghezza 4096)
        canali_completi = True
        for i in range(1, 12):  # Canali 1 a 11
            if data[str(i)].flatten().shape[0] != 4096:
                canali_completi = False
                break

        # Se tutti i canali sono completi, salva il file trovato
        if canali_completi:
            file_trovato = path
            print(f"✅ File trovato: {nome_file}")
            break
    except Exception as e:
        print(f"❌ Errore nel file {nome_file}: {e}")

# Se un file valido è stato trovato, il suo percorso sarà memorizzato in file_trovato
if file_trovato:
    print(f"File valido trovato: {file_trovato}")
else:
    print("Nessun file valido trovato.")

if file_trovato is None:
    print("❌ Nessun file trovato con 4096 elementi per ogni canale.")
    exit()

print(f"✅ File selezionato: {file_trovato}")
data = sio.loadmat(file_trovato)

# === INFO TEMPORALI ===
giorno = int(data['dayOfYear'].flatten()[0])
ora = int(data['iTmOfDay'].flatten()[0])
print(f"Giorno: {giorno}, Ora (slot): {ora} → {(ora * 15) // 60}:{(ora * 15) % 60:02d}")

# === COORDINATE ===
lat = np.nan_to_num(data['12'].flatten(), nan=np.nanmean(data['12'].flatten()))
lon = np.nan_to_num(data['13'].flatten(), nan=np.nanmean(data['13'].flatten()))
lat_grid = lat.reshape(64, 64)
lon_grid = lon.reshape(64, 64)

# === CALCOLO INDICE NELLA MASCHERA ===
# === CALCOLO INDICE NELLA MASCHERA ===
# usa la lista già ordinata per DOY/TIME
indice_file_corrente = file_ordinati.index(os.path.basename(file_trovato))

indice_iniziale = 0
for f in file_ordinati[:indice_file_corrente]:
    dati_temp = sio.loadmat(os.path.join(cartella, f))
    indice_iniziale += dati_temp['1'].shape[0]
    n_immagini_corrente = data['1'].shape[0]  # tipicamente 1 se ogni file ha una sola immagine

# === CARICA LA MASCHERA GIORNO/NOTTE ===
maschera_data = sio.loadmat(maschera_path)
maschera_globale = maschera_data['giorno_maschera'].flatten()
maschera_file = maschera_globale[indice_iniziale:indice_iniziale + n_immagini_corrente]

# === MASCHERA ATTUALE (solo prima immagine del file) ===
maschera_immagine = maschera_file[0]  # 1 → giorno, 0 → notte

# === STAMPA PARTE DELLA MASCHERA PER IL CONTROLLO ===
print("Parte della maschera giorno/notte:")
print(maschera_globale[:20])  # Stampa i primi 20 valori della maschera
print(f"Numero di valori 1 (giorno): {np.sum(maschera_globale == 1)}")
print(f"Numero di valori 0 (notte): {np.sum(maschera_globale == 0)}")

# === PLOT DEI CANALI CON RETTANGOLO GIORNO/NOTTE ===
fig, axs = plt.subplots(3, 4, figsize=(16, 12))
axs = axs.flatten()

# Colore del rettangolo in base alla maschera
colore_rettangolo = 'yellow' if maschera_immagine == 1 else 'blue'  # Cambiato in blu o nero per contrasto

for i in range(1, 12):  # canali 1–11
    canale = data[str(i)].flatten().reshape(64, 64)

    im = axs[i-1].imshow(canale, cmap='bone',
                         extent=[lon_grid.min(), lon_grid.max(), lat_grid.min(), lat_grid.max()])
    axs[i-1].set_title(f'Canale {i}')
    axs[i-1].set_xlabel('Longitudine')
    axs[i-1].set_ylabel('Latitudine')
    fig.colorbar(im, ax=axs[i-1], orientation='vertical', shrink=0.7)

    # Aggiungi un rettangolo con bordo più spesso
    rect = patches.Rectangle(
        (lon_grid.min(), lat_grid.min()),  # angolo inferiore sinistro
        lon_grid.max() - lon_grid.min(),   # larghezza
        lat_grid.max() - lat_grid.min(),   # altezza
        linewidth=4,                       # Aumentato lo spessore della linea
        edgecolor=colore_rettangolo,       # colore del bordo
        facecolor='none'                   # nessun riempimento
    )
    axs[i-1].add_patch(rect)  # Aggiungi il rettangolo all'asse

axs[-1].axis('off')  # Rimuovi subplot vuoto
plt.suptitle(f"Maschera: {'Giorno' if maschera_immagine == 1 else 'Notte'}", fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
import os
import scipy.io as sio
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patches as patches  # Per disegnare il rettangolo

# === PARAMETRI ===
cartella = r'C:\Users\user\Documents\TESI\datigrezzi\DATI_grezzi_2020\OUTPUT_1'
maschera_path = r'C:\Users\user\Documents\TESI\datigrezzi\DATI_grezzi_2020\MASCHERE\giorno_maschera_3h.mat'

# === CARICA LA MASCHERA GIORNO/NOTTE ===
maschera_data = sio.loadmat(maschera_path)
maschera_globale = maschera_data['giorno_maschera'].flatten()

import re

def estrai_doy_time(nome_file):
    match = re.search(r"DOY(\d+)_TIME(\d+)", nome_file)
    if match:
        doy = int(match.group(1))
        time = int(match.group(2))
        return doy, time
    return float('inf'), float('inf')  # Restituisce valori molto alti se non trova match

file_trovato = None

# Ordina i file per DOY e TIME
file_ordinati = sorted(
    [f for f in os.listdir(cartella) if f.endswith('.mat')],
    key=estrai_doy_time
)

# Scorri i file ordinati
for nome_file in file_ordinati:
    path = os.path.join(cartella, nome_file)
    try:
        data = sio.loadmat(path)
        
        # === INFO TEMPORALI ===
        giorno = int(data['dayOfYear'].flatten()[0])
        ora = int(data['iTmOfDay'].flatten()[0])
        print(f"Giorno: {giorno}, Ora (slot): {ora} → {(ora * 15) // 60}:{(ora * 15) % 60:02d}")

        # === COORDINATE ===
        lat = np.nan_to_num(data['12'].flatten(), nan=np.nanmean(data['12'].flatten()))
        lon = np.nan_to_num(data['13'].flatten(), nan=np.nanmean(data['13'].flatten()))
        lat_grid = lat.reshape(64, 64)
        lon_grid = lon.reshape(64, 64)

        # === CALCOLO INDICE NELLA MASCHERA ===
        indice_iniziale = 0
        for f in file_ordinati[:file_ordinati.index(nome_file)]:
            dati_temp = sio.loadmat(os.path.join(cartella, f))
            indice_iniziale += dati_temp['1'].shape[0]
        n_immagini_corrente = data['1'].shape[0]

        # === MASCHERA ATTUALE (solo prima immagine del file) ===
        maschera_file = maschera_globale[indice_iniziale:indice_iniziale + n_immagini_corrente]
        maschera_immagine = maschera_file[0]  # 1 → giorno, 0 → notte

        # === PLOT DEI CANALI CON RETTANGOLO GIORNO/NOTTE ===
        fig, axs = plt.subplots(3, 4, figsize=(16, 12))
        axs = axs.flatten()

        # Colore del rettangolo in base alla maschera
        colore_rettangolo = 'yellow' if maschera_immagine == 1 else 'blue'  # Cambiato in blu o nero per contrasto

        for i in range(1, 12):  # canali 1–11
            canale = data[str(i)].flatten().reshape(64, 64)

            im = axs[i-1].imshow(canale, cmap='bone',
                                 extent=[lon_grid.min(), lon_grid.max(), lat_grid.min(), lat_grid.max()])
            axs[i-1].set_title(f'Canale {i}')
            axs[i-1].set_xlabel('Longitudine')
            axs[i-1].set_ylabel('Latitudine')
            fig.colorbar(im, ax=axs[i-1], orientation='vertical', shrink=0.7)

            # Aggiungi un rettangolo con bordo più spesso
            rect = patches.Rectangle(
                (lon_grid.min(), lat_grid.min()),  # angolo inferiore sinistro
                lon_grid.max() - lon_grid.min(),   # larghezza
                lat_grid.max() - lat_grid.min(),   # altezza
                linewidth=4,                       # Aumentato lo spessore della linea
                edgecolor=colore_rettangolo,       # colore del bordo
                facecolor='none'                   # nessun riempimento
            )
            axs[i-1].add_patch(rect)  # Aggiungi il rettangolo all'asse

        axs[-1].axis('off')  # Rimuovi subplot vuoto
        plt.suptitle(f"Maschera: {'Giorno' if maschera_immagine == 1 else 'Notte'}", fontsize=16)
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])

        # Salva la figura per ogni file (opzionale)
        #plt.savefig(f'{os.path.splitext(nome_file)[0]}_canali_con_maschera.png')
        plt.show()

    except Exception as e:
        print(f"Errore nel file {nome_file}: {e}")

In [ ]:
import os
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import re

# === PARAMETRI ===
cartella = r'C:\Users\user\Documents\TESI\datigrezzi\DATI_grezzi_2020\OUTPUT_1'
maschera_path = r'C:\Users\user\Documents\TESI\datigrezzi\DATI_grezzi_2020\MASCHERE\giorno_maschera_3h.mat'

# === CARICA LA MASCHERA GLOBALE ===
maschera_data = sio.loadmat(maschera_path)
maschera_globale = maschera_data['giorno_maschera'].flatten()

# === COSTANTI ===
DIM_IMMAGINE = 4096  # 64x64
indice_attuale = 0

# === Funzione per ordinare i file ===
def estrai_doy_time(nome_file):
    match = re.search(r"DOY(\d+)_TIME(\d+)", nome_file)
    if match:
        doy = int(match.group(1))
        time = int(match.group(2))
        return doy, time
    return float('inf'), float('inf')

# === Ordina i file ===
file_ordinati = sorted(
    [f for f in os.listdir(cartella) if f.endswith('.mat')],
    key=estrai_doy_time
)

# === CICLO SUI FILE ===
for nome_file in file_ordinati:
    path = os.path.join(cartella, nome_file)
    try:
        data = sio.loadmat(path)

        giorno = int(data['dayOfYear'].flatten()[0])
        ora = int(data['iTmOfDay'].flatten()[0])
        print(f"\n--- {nome_file} ---")
        print(f"Giorno: {giorno}, Ora slot: {ora} → {(ora * 15) // 60}:{(ora * 15) % 60:02d}")

        # === Estrai e analizza la maschera per questa immagine ===
        maschera_img = maschera_globale[indice_attuale:indice_attuale + DIM_IMMAGINE]
        num_giorno = np.sum(maschera_img == 1)
        num_notte = np.sum(maschera_img == 0)

        print(f"→ Pixel giorno: {num_giorno}, notte: {num_notte}")

        # Colore rettangolo: giallo se più del 50% è giorno
        colore_rettangolo = 'yellow' if num_giorno >= DIM_IMMAGINE // 2 else 'blue'
        label_maschera = 'Giorno' if colore_rettangolo == 'yellow' else 'Notte'

        # === Lat e Lon ===
        lat = np.nan_to_num(data['12'].flatten(), nan=np.nanmean(data['12'].flatten())).reshape(64, 64)
        lon = np.nan_to_num(data['13'].flatten(), nan=np.nanmean(data['13'].flatten())).reshape(64, 64)

        lat = np.rot90(lat, k=1)
        lon = np.rot90(lon, k=1)

        # === Plot ===
        fig, axs = plt.subplots(3, 4, figsize=(16, 12))
        axs = axs.flatten()

        for i in range(1, 12):  # canali da 1 a 11
            canale = data[str(i)].flatten().reshape(64, 64)
            canale = np.rot90(canale, k=1)  # ruota 90° in senso orario

            im = axs[i-1].imshow(canale, cmap='bone',
                                 extent=[lon.min(), lon.max(), lat.min(), lat.max()])

            axs[i-1].set_title(f'Canale {i}')
            axs[i-1].set_xlabel('Longitudine')
            axs[i-1].set_ylabel('Latitudine')
            fig.colorbar(im, ax=axs[i-1], orientation='vertical', shrink=0.7)

            # Aggiungi rettangolo
            rect = patches.Rectangle(
                (lon.min(), lat.min()),
                lon.max() - lon.min(),
                lat.max() - lat.min(),
                linewidth=4,
                edgecolor=colore_rettangolo,
                facecolor='none'
            )
            axs[i-1].add_patch(rect)

        # === Canale DPR ===
        try:
            canale_dpr = data['dpr'].flatten().reshape(64, 64)
            canale_dpr = np.rot90(canale_dpr, k=1)

            im = axs[11].imshow(canale_dpr, cmap='Blues',
                                extent=[lon.min(), lon.max(), lat.min(), lat.max()])

            axs[11].set_title('DPR')
            axs[11].set_xlabel('Longitudine')
            axs[11].set_ylabel('Latitudine')
            fig.colorbar(im, ax=axs[11], orientation='vertical', shrink=0.7)

            rect = patches.Rectangle(
                (lon.min(), lat.min()),
                lon.max() - lon.min(),
                lat.max() - lat.min(),
                linewidth=4,
                edgecolor=colore_rettangolo,
                facecolor='none'
            )
            axs[11].add_patch(rect)

        except Exception as e:
            axs[11].axis('off')
            print(f"⚠️ Impossibile caricare il canale DPR: {e}")

        plt.suptitle(f"{nome_file} | Maschera: {label_maschera} | Giorno: {num_giorno}, Notte: {num_notte}", fontsize=16)
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.show()

        # Aggiorna l'indice per la prossima immagine
        indice_attuale += DIM_IMMAGINE

    except Exception as e:
        print(f"Errore nel file {nome_file}: {e}")